# Transformer EN→DE 학습 (Google Colab)

**런타임 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 1. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. 저장소 클론 및 의존성 설치

In [ ]:
import os

if not os.path.exists('/content/MyTransformer'):
    !git clone https://github.com/youngpark-POS/MyTransformer.git
else:
    print('이미 클론됨, 최신 코드로 업데이트...')
    !git -C /content/MyTransformer pull

%cd /content/MyTransformer

In [ ]:
import os
# subprocess(!python ...)가 src 레이아웃 패키지를 찾을 수 있도록 PYTHONPATH 설정
# NOTE: pip install -e . 의 editable finder는 Colab에서 transformer 서브패키지(data/model)를
#       노출하지 못해 ModuleNotFoundError를 일으키므로 사용하지 않고 PYTHONPATH만 쓴다.
os.environ['PYTHONPATH'] = '/content/MyTransformer/src'

# 혹시 이전 실행에서 남은 editable install이 PYTHONPATH를 가로채지 않도록 제거(있을 때만)
!pip uninstall -y transformer -q 2>/dev/null

# torch는 Colab에 기설치 — datasets, sacrebleu, tqdm만 추가 설치
!pip install -q datasets sacrebleu tqdm

## 3. 학습

In [ ]:
# 첫 실행 시 HuggingFace에서 Multi30k 다운로드 + vocab 캐시 생성 (수 분 소요)
!python -m transformer.train --epochs 10

## 4. 체크포인트를 Google Drive에 저장

> Colab 세션이 끊기면 `/content` 내용이 사라집니다. Drive에 백업해두세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os
dst = '/content/drive/MyDrive/MyTransformer'
os.makedirs(dst, exist_ok=True)
shutil.copy('checkpoints/best.pt', dst + '/best.pt')
print('저장 완료:', dst + '/best.pt')

## 5. 번역 추론

In [ ]:
!python -m transformer.translate --text "a man is riding a bike"

In [ ]:
# beam search (beam=5)
!python -m transformer.translate --text "a man is riding a bike" --beam 5

## 6. (선택) Drive에서 체크포인트 불러와 추론

세션이 재시작된 경우, Drive에서 체크포인트를 복원해 추론만 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('checkpoints', exist_ok=True)
shutil.copy('/content/drive/MyDrive/MyTransformer/best.pt', 'checkpoints/best.pt')
print('체크포인트 복원 완료')

In [ ]:
!python -m transformer.translate --text "two dogs are playing in the park" --beam 5

## 7. 모델 정확도 평가 (테스트셋 100쌍)

Multi30k **테스트 split**에서 EN→DE 평행 코퍼스 100쌍을 가져와, 모델이 영어를 번역하게 하고
정답 독일어와 비교해 정확도를 측정한다. (모델이 학습한 것과 같은 도메인의 사람 번역 정답이라 공정한 평가가 된다.)

- **BLEU**: 기계번역 표준 지표 (0~100, 높을수록 좋음)
- **완전 일치 정확도**: 예측 문장이 정답과 전체가 똑같은 비율
- **단어 단위 정확도**: 정답 단어 중 예측에 포함된 비율의 평균

> 공정 비교를 위해 정답에도 모델과 동일한 토큰화/복원 규칙(`detokenize(tokenize(...))`)을 적용한다.

In [ ]:
import sys
sys.path.insert(0, '/content/MyTransformer/src')
from pathlib import Path
from transformer.translate import load_for_inference, translate
from transformer.data.dataset import _load_pairs
from transformer.data.tokenizer import tokenize, detokenize
from transformer.utils import resolve_device

device = resolve_device('cuda')
model, src_vocab, tgt_vocab, cfg = load_for_inference(Path('checkpoints/best.pt'), device)

# Multi30k 테스트셋에서 EN->DE 100쌍 (학습 도메인의 사람 번역 정답)
pairs = _load_pairs('test', 'en', 'de')[:100]
print(f'모델 로드 완료 / 평가 쌍: {len(pairs)}개')

In [ ]:
from collections import Counter
import sacrebleu

BEAM = 1  # 1=greedy(빠름), 5=beam search(느리지만 품질↑)

def normalize(text):
    # 모델 출력과 같은 토큰화/복원 규칙을 정답에도 적용해 공정하게 비교
    return detokenize(tokenize(text))

def word_acc(hyp, ref):
    # 정답 토큰 중 예측에 등장한 비율 (다중집합 기준)
    h, r = Counter(hyp.split()), ref.split()
    if not r:
        return 0.0
    match = 0
    for t in r:
        if h[t] > 0:
            h[t] -= 1
            match += 1
    return match / len(r)

hyps, refs, rows = [], [], []
for en, de in pairs:
    hyp = translate(en, model, src_vocab, tgt_vocab, device, beam_size=BEAM, max_len=cfg.model.max_len)
    ref = normalize(de)
    hyps.append(hyp); refs.append(ref); rows.append((en, ref, hyp))

bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
n_exact = sum(h == r for h, r in zip(hyps, refs))
exact = n_exact / len(hyps)
token_acc = sum(word_acc(h, r) for h, r in zip(hyps, refs)) / len(hyps)

print(f'BLEU             : {bleu:.2f}')
print(f'완전 일치 정확도 : {exact*100:.1f}%  ({n_exact}/{len(hyps)})')
print(f'단어 단위 정확도 : {token_acc*100:.1f}%')

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 60)
df = pd.DataFrame(rows, columns=['EN (입력)', 'DE (정답)', 'DE (모델 예측)'])
df.head(15)